# Zadanie 3 — Cassandra: model danych i zapytania

### Tabela:
```sql
CREATE TABLE iot_data (
    device_id TEXT,
    event_time TIMESTAMP,
    temperature DOUBLE,
    PRIMARY KEY (device_id, event_time)
);
```

In [15]:
from cassandra.cluster import Cluster

cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print('Polaczono z Cassandra')

Polaczono z Cassandra


In [16]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
session.set_keyspace('lab_keyspace')
print('Keyspace ustawiony')

Keyspace ustawiony


## Tworzenie tabeli i wstawianie danych

In [17]:
session.execute('DROP TABLE IF EXISTS iot_data')

session.execute("""
    CREATE TABLE IF NOT EXISTS iot_data (
        device_id TEXT,
        event_time TIMESTAMP,
        temperature DOUBLE,
        PRIMARY KEY (device_id, event_time)
    )
""")
print('Tabela utworzona')

Tabela utworzona


In [18]:
rows = [
    ('dev1', '2024-01-01 10:00:00', 22.1),
    ('dev1', '2024-01-01 10:05:00', 22.5),
    ('dev1', '2024-01-01 10:10:00', 23.0),
    ('dev1', '2024-01-01 10:15:00', 23.4),
    ('dev1', '2024-01-01 10:20:00', 24.0),
    ('dev2', '2024-01-01 10:00:00', 18.1),
    ('dev2', '2024-01-01 10:05:00', 18.5),
    ('dev2', '2024-01-01 10:10:00', 19.0),
    ('dev2', '2024-01-01 10:15:00', 19.4),
    ('dev2', '2024-01-01 10:20:00', 20.0),
    ('dev3', '2024-01-01 10:00:00', 25.1),
    ('dev3', '2024-01-01 10:05:00', 25.5),
    ('dev3', '2024-01-01 10:10:00', 26.0),
    ('dev3', '2024-01-01 10:15:00', 26.4),
    ('dev3', '2024-01-01 10:20:00', 27.0),
]

for device_id, event_time, temperature in rows:
    session.execute(
        'INSERT INTO iot_data (device_id, event_time, temperature) VALUES (%s, %s, %s)',
        (device_id, event_time, temperature)
    )

print(f'Wstawiono {len(rows)} rekordow')

Wstawiono 15 rekordow


---
## Polecenie 1a — wszystkie odczyty dla urządzenia dev1

```cql
SELECT * FROM iot_data WHERE device_id = 'dev1';
```
Zapytanie działa poprawnie — podajemy partition key `device_id`, więc Cassandra od razu wie, na której partycji szukać danych.

In [19]:
result = session.execute("SELECT * FROM iot_data WHERE device_id = 'dev1'")

print('Odczyty dla dev1:')
print(f'{"device_id":<10} {"event_time":<25} {"temperature"}')
print('-' * 50)
for row in result:
    print(f'{row.device_id:<10} {str(row.event_time):<25} {row.temperature}')

Odczyty dla dev1:
device_id  event_time                temperature
--------------------------------------------------
dev1       2024-01-01 10:00:00       22.1
dev1       2024-01-01 10:05:00       22.5
dev1       2024-01-01 10:10:00       23.0
dev1       2024-01-01 10:15:00       23.4
dev1       2024-01-01 10:20:00       24.0


---
## Polecenie 1b — pomiary dla dev1 od 10:05:00 do 10:15:00

In [20]:
result = session.execute("""
    SELECT * FROM iot_data
    WHERE device_id = 'dev1'
      AND event_time >= '2024-01-01 10:05:00'
      AND event_time <= '2024-01-01 10:15:00'
""")

print('Pomiary dla dev1 od 10:05 do 10:15:')
print(f'{"device_id":<10} {"event_time":<25} {"temperature"}')
print('-' * 50)
for row in result:
    print(f'{row.device_id:<10} {str(row.event_time):<25} {row.temperature}')

Pomiary dla dev1 od 10:05 do 10:15:
device_id  event_time                temperature
--------------------------------------------------
dev1       2024-01-01 10:05:00       22.5
dev1       2024-01-01 10:10:00       23.0
dev1       2024-01-01 10:15:00       23.4


---
## Polecenie 2 — Dlaczego nie można zapytać tylko po `event_time` bez `device_id`?

`device_id` jest partition key — Cassandra wyznacza z niego (przez hash) węzeł klastra, na którym leży dana partycja. Bez niego baza nie wiedziałaby, które węzły odpytać, co wymuszałoby pełny skan całego klastra — stąd Cassandra po prostu odrzuca takie zapytanie.

In [21]:
cluster.shutdown()
print('Polaczenie zamkniete')

Polaczenie zamkniete
